# NB6057CEM — Artificial Neural Networks (CW1 T1–T3 Real-Data Training & Zenodo Acquisition)
**Project:** Robust Emergency Siren Classification Under Urban Noise: A Comparative Study of MLP and CNN Architectures

This self-contained notebook is engineered for **VS Code with a Google Colab kernel**, browser **Google Colab Pro**, or a local environment:
1. **Runtime Verification:** Verifies whether execution is genuinely inside a Google Colab Linux VM before touching `/content` paths, and protects the local Windows `C:` drive from 6 GB downloads.
2. **Automatic Cloud Workspace Bootstrap:** Because a VS Code Colab kernel runs on a remote VM without direct access to local `D:\ANN-Siren-Detection` files, Cell 2 automatically unpacks all verified project modules (`configs/`, `src/`, `scripts/`, `requirements.txt`) into `/content/ANN-Siren-Detection`.
3. **Resumable Official Zenodo Download + Google Drive Persistence:** Downloads `UrbanSound8K.tar.gz` (`6,023,741,708` bytes, MD5 `9aa69802bbf37fb986f71ec1483a196e`) from `https://zenodo.org/records/1203745` with HTTP `Range` resume support, persists the verified archive to Google Drive so you never have to re-download 6 GB, and safely extracts it.
4. **Full T1–T3 Pipeline Execution:** Runs `audit_dataset.py`, `prepare_dataset.py`, `run_numpy_mlp_t2.py`, and `run_competing_models_t3.py`.

In [ ]:
# CELL 1: Verify Runtime Environment (Colab Cloud VM vs. Local Machine) & Mount Google Drive
import os
import platform
import sys
from pathlib import Path

in_colab_module = "google.colab" in sys.modules
has_colab_env = any(k in os.environ for k in ("COLAB_RELEASE_TAG", "COLAB_GPU", "COLAB_JUPYTER_IP"))
is_linux_content = platform.system().lower() == "linux" and Path("/content").is_dir()
IS_COLAB_RUNTIME = bool(in_colab_module or has_colab_env or is_linux_content)

print(f"Platform         : {platform.system()} {platform.release()}")
print(f"Python Executable: {sys.executable}")
print(f"Is Colab Runtime : {IS_COLAB_RUNTIME}")

GDRIVE_CACHE_DIR = None
GDRIVE_ARTIFACTS_DIR = None

if IS_COLAB_RUNTIME:
    PROJECT_ROOT = Path(os.environ.get("ANN_PROJECT_ROOT", "/content/ANN-Siren-Detection")).resolve()
    DOWNLOAD_SCRATCH_DIR = Path("/content/urbansound8k_dl").resolve()
    # Mount Google Drive for persistent caching of the 6.02 GB archive and trained checkpoints
    if os.environ.get("MOUNT_GDRIVE", "1") == "1":
        try:
            from google.colab import drive  # type: ignore
            mount_point = os.environ.get("GDRIVE_MOUNT_POINT", "/content/drive")
            if not Path(mount_point, "MyDrive").is_dir():
                drive.mount(mount_point)
            gdrive_root = Path(mount_point) / "MyDrive" / "ANN-Siren-Detection"
            GDRIVE_CACHE_DIR = gdrive_root / "cache"
            GDRIVE_ARTIFACTS_DIR = gdrive_root / "artifacts"
            GDRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
            GDRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
            print(f"Google Drive Cache Dir    : {GDRIVE_CACHE_DIR}")
            print(f"Google Drive Artifacts Dir: {GDRIVE_ARTIFACTS_DIR}")
        except Exception as exc:
            print(f"[WARN] Google Drive mount skipped or unavailable ({exc}). Using /content storage only.")
else:
    # Running on local machine (e.g. D:\ANN-Siren-Detection) — NEVER use /content paths
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    DOWNLOAD_SCRATCH_DIR = PROJECT_ROOT / "data" / "raw"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Active Project Root       : {PROJECT_ROOT}")

In [ ]:
# CELL 2: Bootstrap Project Files into Colab Runtime (No Local File Assumption)
import base64
import hashlib
import io
import zipfile

EXPECTED_BUNDLE_SHA256 = "a8ce7c96430483503595bdbbe5c294a6075af8299fd73df19c32c51c1d1ee7ca"
EMBEDDED_PROJECT_BUNDLE_B64 = "..."

required_markers = [
    PROJECT_ROOT / "configs" / "config.yaml",
    PROJECT_ROOT / "configs" / "data_config.yaml",
    PROJECT_ROOT / "configs" / "model_config.yaml",
    PROJECT_ROOT / "src" / "models.py",
    PROJECT_ROOT / "scripts" / "acquire_dataset.py",
    PROJECT_ROOT / "scripts" / "audit_dataset.py",
    PROJECT_ROOT / "scripts" / "prepare_dataset.py",
    PROJECT_ROOT / "scripts" / "run_numpy_mlp_t2.py",
    PROJECT_ROOT / "scripts" / "run_competing_models_t3.py",
]

if IS_COLAB_RUNTIME or not all(p.is_file() for p in required_markers):
    raw_zip = base64.b64decode(EMBEDDED_PROJECT_BUNDLE_B64)
    actual_sha = hashlib.sha256(raw_zip).hexdigest()
    assert actual_sha == EXPECTED_BUNDLE_SHA256, f"Bundle checksum mismatch: {actual_sha}"
    with zipfile.ZipFile(io.BytesIO(raw_zip), "r") as zf:
        zf.extractall(PROJECT_ROOT)
    print(f"[OK] Bootstrapped {len(required_markers)}+ project files into {PROJECT_ROOT}")
else:
    print(f"[OK] Local project files verified in {PROJECT_ROOT}")

for rel_dir in [
    "data/raw",
    "data/processed",
    "data/splits",
    "checkpoints",
    "results/metrics",
    "results/figures",
    "reports",
]:
    (PROJECT_ROOT / rel_dir).mkdir(parents=True, exist_ok=True)

for marker in required_markers:
    assert marker.is_file(), f"Missing required project file: {marker}"
print("[OK] All required project scripts, configs, and source modules are ready.")

In [ ]:
# CELL 3: Verify Dependencies, Hardware (GPU/CPU), and Zenodo Network Connectivity
import subprocess

if IS_COLAB_RUNTIME:
    # Colab already pre-installs torch, torchaudio, numpy, pandas, scipy, scikit-learn, librosa, matplotlib
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "soundfile", "seaborn"])

from src import get_environment_info, load_config, select_device
from scripts.acquire_dataset import check_zenodo_source

cfg = load_config()
env_report = get_environment_info(cfg)
device_info = select_device("auto")

print(f"Python Version : {env_report.python_version}")
print(f"PyTorch Version: {env_report.torch_version}")
print(f"Selected Device: {device_info.selected_device} (CUDA available: {device_info.cuda_available})")
if device_info.cuda_device_name:
    print(f"GPU Device Name: {device_info.cuda_device_name}")

zenodo_status = check_zenodo_source()
print("\nZenodo Official Source Check (https://zenodo.org/records/1203745):")
print(f"  API Accessible       : {zenodo_status['api_accessible']} (HTTP {zenodo_status['http_status_api']})")
print(f"  Download Accessible  : {zenodo_status['download_accessible']} (HTTP {zenodo_status['http_status_download']})")
print(f"  Supports Range Resume: {zenodo_status['supports_range_resume']}")
print(f"  Expected Archive Size: {zenodo_status['expected_bytes']:,} bytes (6.02 GB)")
print(f"  Expected Archive MD5 : {zenodo_status['expected_md5']}")

In [ ]:
# CELL 4: Resumable Zenodo Download, Integrity Verification, Google Drive Persistence & Safe Extraction
from scripts.acquire_dataset import main as acquire_main

acquire_cli_args = [
    "--download",
    "--download-dir",
    str(DOWNLOAD_SCRATCH_DIR),
    "--target-raw-dir",
    str(PROJECT_ROOT / "data" / "raw"),
]
if GDRIVE_CACHE_DIR is not None:
    acquire_cli_args.extend(["--gdrive-cache-dir", str(GDRIVE_CACHE_DIR)])

rc_acquire = acquire_main(acquire_cli_args)
assert rc_acquire == 0, f"Dataset acquisition failed with exit code {rc_acquire}"

In [ ]:
# CELL 5: Execute Real-Data Audit (T1b), Split/Scaler Prep (T1c), NumPy MLP (T2), and Competing Models (T3)
import subprocess
import json

pipeline_scripts = [
    "scripts/audit_dataset.py",
    "scripts/prepare_dataset.py",
    "scripts/run_numpy_mlp_t2.py",
    "scripts/run_competing_models_t3.py",
]

for script_rel in pipeline_scripts:
    print("\n" + "=" * 78)
    print(f"RUNNING: python {script_rel}")
    print("=" * 78, flush=True)
    proc = subprocess.run([sys.executable, script_rel], cwd=str(PROJECT_ROOT), check=False)
    if proc.returncode != 0:
        raise RuntimeError(f"{script_rel} exited with non-zero code {proc.returncode}")

print("\n" + "=" * 78)
print("REAL-DATA PIPELINE EXECUTION SUMMARY (T1 - T3)")
print("=" * 78)
for metric_file in [
    "results/metrics/dataset_audit.json",
    "results/metrics/t2_numpy_mlp_Metrics.json",
    "results/metrics/t3_competing_models_summary.json",
]:
    p = PROJECT_ROOT / metric_file
    if not p.is_file():
        p = PROJECT_ROOT / metric_file.lower()
    if p.is_file():
        print(f"[VERIFIED] {p.relative_to(PROJECT_ROOT)} ({p.stat().st_size:,} bytes)")

In [ ]:
# CELL 6: Persist Generated Results/Checkpoints to Google Drive & Package Artifact ZIP for Local Sync
import shutil

artifact_dirs = [
    "checkpoints",
    "results/metrics",
    "results/figures",
    "data/splits",
    "data/processed",
    "data/raw/DATASET_PROVENANCE.json",
]

staging_dir = PROJECT_ROOT / "_artifact_bundle"
if staging_dir.exists():
    shutil.rmtree(staging_dir)

for rel_item in artifact_dirs:
    src_item = PROJECT_ROOT / rel_item
    if src_item.is_dir():
        shutil.copytree(src_item, staging_dir / rel_item, dirs_exist_ok=True)
        if GDRIVE_ARTIFACTS_DIR is not None:
            shutil.copytree(src_item, GDRIVE_ARTIFACTS_DIR / rel_item, dirs_exist_ok=True)
    elif src_item.is_file():
        dest_file = staging_dir / rel_item
        dest_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_item, dest_file)
        if GDRIVE_ARTIFACTS_DIR is not None:
            gd_file = GDRIVE_ARTIFACTS_DIR / rel_item
            gd_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_item, gd_file)

zip_base = PROJECT_ROOT / "ann_siren_t1_t2_t3_artifacts"
zip_file = shutil.make_archive(str(zip_base), "zip", root_dir=staging_dir)
shutil.rmtree(staging_dir)
print(f"[OK] Packaged all real-data artifacts into: {zip_file}")

if GDRIVE_ARTIFACTS_DIR is not None:
    gd_zip = GDRIVE_ARTIFACTS_DIR / Path(zip_file).name
    shutil.copy2(zip_file, gd_zip)
    print(f"[OK] Saved persistent copy of artifact bundle to Google Drive: {gd_zip}")